# Module 5: RAG Pipeline with Azure DocumentDB (Node.js completed reference)

In [ ]:
const { MongoClient } = require("mongodb");const uri = process.env.DOCUMENTDB_CONNECTION_STRING;if (!uri) throw new Error("Set DOCUMENTDB_CONNECTION_STRING before running this notebook.");const client = new MongoClient(uri);await client.connect();const db = client.db("docdbworkshop");const collection = db.collection("rag_chunks");await db.command({ ping: 1 });

In [ ]:
const ragDocs = [  { _id: "rag-001", sourceId: "search-module", title: "Vector search", chunk: "Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.", url: "module-4-search", tags: ["vector", "search"], embedding: [0.92, 0.80, 0.18] },  { _id: "rag-002", sourceId: "search-module", title: "Full-text search", chunk: "Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.", url: "module-4-search", tags: ["full-text", "bm25"], embedding: [0.20, 0.12, 0.94] },  { _id: "rag-003", sourceId: "search-module", title: "Hybrid search", chunk: "Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.", url: "module-4-search", tags: ["hybrid", "rrf"], embedding: [0.76, 0.70, 0.42] },  { _id: "rag-004", sourceId: "rag-module", title: "Grounded generation", chunk: "A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.", url: "module-5-rag", tags: ["rag", "generation"], embedding: [0.84, 0.73, 0.34] }];await collection.deleteMany({});await collection.insertMany(ragDocs);await collection.countDocuments({});

In [ ]:
await db.command({ createIndexes: "rag_chunks", indexes: [{ name: "idx_chunk_embedding_diskann", key: { embedding: "cosmosSearch" }, cosmosSearchOptions: { kind: "vector-diskann", dimensions: 3, similarity: "COS", maxDegree: 32, lBuild: 64 } }] });await db.command({ createSearchIndexes: "rag_chunks", indexes: [{ name: "idx_chunk_fts", definition: { mappings: { dynamic: false, fields: { chunk: { type: "string" } } } } }] });

In [ ]:
const question = "How does DocumentDB retrieve context for RAG?";const questionVector = [0.83, 0.74, 0.33];const vectorContext = await collection.aggregate([{ $search: { cosmosSearch: { path: "embedding", vector: questionVector, k: 3 } } }, { $project: { _id: 1, title: 1, chunk: 1, url: 1, score: { $meta: "searchScore" } } }]).toArray();vectorContext;

In [ ]:
const keywordContext = await collection.aggregate([{ $search: { index: "idx_chunk_fts", text: { query: question, path: "chunk" } } }, { $limit: 3 }, { $project: { _id: 1, title: 1, chunk: 1, url: 1, score: { $meta: "searchScore" } } }]).toArray();function rrf(lists, k = 60, topN = 3) {  const scores = new Map(); const docsById = new Map();  for (const list of lists) {    list.forEach((doc, rank) => { const id = doc._id.toString(); docsById.set(id, doc); scores.set(id, (scores.get(id) ?? 0) + 1 / (k + rank + 1)); });  }  return [...scores.entries()].sort((a, b) => b[1] - a[1]).slice(0, topN).map(([id, score]) => ({ ...docsById.get(id), rrfScore: score }));}const hybridContext = rrf([keywordContext, vectorContext]);hybridContext;

In [ ]:
const contextBlock = hybridContext.map((doc, i) => `[${i + 1}] ${doc.title}\n${doc.chunk}\nSource: ${doc.url}`).join("\n\n");const groundedPrompt = `You are a helpful assistant for an Azure DocumentDB workshop.Answer the user's question using only the context below. If the answer is not present, say you do not know.<context>${contextBlock}</context>Question: ${question}`;console.log(groundedPrompt);